# Create User and Client Profiles

This notebook demonstrates how to create user and client profiles using the modern Person/Actor architecture with Pydantic models, bidirectional User-Client assignment, and YAML persistence via HydraConfigManager.

**Test case**: Hillcrest Children & Family Center as a Client of Masai Interactive, managed by Dheeraj Chand (Siege Analytics).

## What this shows
Creating, persisting, and branding user + client profiles (logos, fonts, colors).

## Why it matters
Branded output (PDF, PPTX, Slides) pulls style from the profile system.

## Prereqs
- `pip install 'siege-utilities[profiles,reporting]'`

## Next
- `06_Report_Generation.ipynb` and `12_PowerPoint_Generation.ipynb` consume these profiles.


> **See also:** Part 2 of this notebook (Person / Actor) covers the full Person/Actor architecture in depth (Organizations, Collaborations, credential management). This notebook focuses on the practical workflow of creating and managing profiles.


In [ ]:
# Import required modules — modern Person/Actor architecture
from pathlib import Path
from siege_utilities.config import User, Client, BrandingConfig, ReportPreferences
from siege_utilities.config import HydraConfigManager
import siege_utilities as su

# Initialize logging
su.configure_shared_logging(level="INFO")

# --- Output configuration ---
OUTPUT_DIR = Path("output") / "notebook_02"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

su.log_info(f"Output directory: {OUTPUT_DIR}  (exists={OUTPUT_DIR.exists()})")
su.log_info("Starting profile creation demo")

## 1. Creating a User Profile

Let's create a comprehensive user profile with all available options:

In [ ]:
import os
from pathlib import Path

# Define user identity variables up front
username = "dheerajchand"
full_name = "Dheeraj Chand"
email = "dheeraj@siegeanalytics.com"

# Create a comprehensive user profile using the modern User model
user = User(
    # Core Identity (from Person base)
    person_id=username,
    name=full_name,
    email=email,
    
    # User-specific fields
    username=username,
    github_login=username,
    role="admin",
    
    # Preferences (cross-platform path)
    preferred_download_directory=str(Path.home() / "Downloads" / "siege_utilities"),
    default_output_format="pdf",
    preferred_map_style="carto-positron",
    default_color_scheme="viridis",
    
    # Technical Preferences
    default_dpi=300,
    default_figure_size=(12, 8),
    enable_logging=True,
    log_level="INFO",
    
    # API Keys (optional — will be populated from 1Password)
    google_analytics_key="",
    facebook_business_key="",
    census_api_key="",
    
    # Database Preferences
    default_database="postgresql",
    postgresql_connection="",
    duckdb_path=""
)

su.log_info("User Created Successfully!")
su.log_info(f"Person ID: {user.person_id}")
su.log_info(f"Name: {user.name}")
su.log_info(f"Email: {user.email}")
su.log_info(f"Username: {user.username}")
su.log_info(f"Role: {user.role}")
su.log_info(f"Download Directory: {user.preferred_download_directory}")
su.log_info(f"Output Format: {user.default_output_format}")
su.log_info(f"Color Scheme: {user.default_color_scheme}")
su.log_info(f"Map Style: {user.preferred_map_style}")
su.log_info(f"DPI: {user.default_dpi}")
su.log_info(f"Figure Size: {user.default_figure_size}")

## 2. Creating Branding Configuration

Now let's create a branding configuration for Hillcrest Children & Family Center:

In [ ]:
# Create branding configuration for Hillcrest with hex color validation
hillcrest_branding = BrandingConfig(
    # Colors (must be valid hex codes) — Hillcrest brand palette
    primary_color="#1a4d2e",      # Forest green
    secondary_color="#4a7c59",    # Sage green
    accent_color="#f4a261",       # Warm orange
    text_color="#2d3436",         # Dark gray
    background_color="#ffffff",   # White
    
    # Typography
    primary_font="Helvetica",
    secondary_font="Georgia",
    
    # Logo (optional — will be set when logo file is available)
    logo_path=None,
    logo_width=None,
    logo_height=None,
    
    # Layout dimensions
    header_height=50,
    footer_height=25,
    margin_top=25,
    margin_bottom=25,
    
    # Chart styling
    chart_color_palette="viridis"
)

su.log_info("Hillcrest Branding Configuration Created!")
su.log_info(f"Primary Color: {hillcrest_branding.primary_color}")
su.log_info(f"Secondary Color: {hillcrest_branding.secondary_color}")
su.log_info(f"Accent Color: {hillcrest_branding.accent_color}")
su.log_info(f"Primary Font: {hillcrest_branding.primary_font}")
su.log_info(f"Secondary Font: {hillcrest_branding.secondary_font}")
su.log_info(f"Header Height: {hillcrest_branding.header_height}px")
su.log_info(f"Footer Height: {hillcrest_branding.footer_height}px")

## 3. Creating a Client Profile

Now let's create Hillcrest Children & Family Center as a Client of Masai Interactive, with Dheeraj as the assigned user:

In [ ]:
# Create Hillcrest as a Client using the modern Client model
client = Client(
    # Core Identity (from Person base)
    person_id="hillcrest_client",
    name="Hillcrest Children & Family Center",
    email="info@hillcrestchildren.org",
    website="https://www.hillcrestchildren.org",
    address="Washington, DC",
    
    # Client-specific fields
    client_code="HILL",          # Must be uppercase, 2-10 characters
    industry="Nonprofit - Children & Family Services",
    project_count=1,
    client_status="active",      # active, inactive, or archived
    
    # Organization context — Hillcrest is a client of Masai Interactive
    organizations=["masai_interactive"],
    primary_organization="masai_interactive",
    
    # Client-specific configurations
    branding_config=hillcrest_branding,
    report_preferences=ReportPreferences()  # Use defaults
)

# Demonstrate User-Client bidirectional assignment
client.assign_user(user.person_id)
client.set_primary_user(user.person_id)
user.assign_client(client.client_code)
user.set_primary_client(client.client_code)

su.log_info("Client Created Successfully!")
su.log_info(f"Person ID: {client.person_id}")
su.log_info(f"Name: {client.name}")
su.log_info(f"Client Code: {client.client_code}")
su.log_info(f"Industry: {client.industry}")
su.log_info(f"Project Count: {client.project_count}")
su.log_info(f"Status: {client.client_status}")
su.log_info(f"Email: {client.email}")
su.log_info(f"Website: {client.website}")
su.log_info(f"Address: {client.address}")
su.log_info(f"Organization: {client.primary_organization}")
su.log_info(f"Brand Primary Color: {client.branding_config.primary_color}")
su.log_info(f"Brand Primary Font: {client.branding_config.primary_font}")
su.log_info(f"Assigned Users: {client.get_assigned_users()}")
su.log_info(f"Primary User: {client.primary_user}")
su.log_info(f"User's Assigned Clients: {user.get_assigned_clients()}")
su.log_info(f"User's Primary Client: {user.primary_client}")

## 4. Validation Examples

Let's see how the validation system works with some examples:

In [ ]:
# Test email validation on modern User model
su.log_info("Testing Email Validation:")
try:
    valid_email_user = User(
        person_id="test-user", name="Test User", email="valid@example.com", username="testuser"
    )
    su.log_info("   Valid email: valid@example.com")
except ValueError as e:
    su.log_error(f"   Error: {e}")

try:
    invalid_email_user = User(
        person_id="test-user2", name="Test User", email="invalid-email", username="testuser2"
    )
    su.log_info("   Invalid email handled gracefully")
except ValueError as e:
    su.log_error(f"   Validation error: {e}")

# Test color validation
su.log_info("Testing Color Validation:")
try:
    valid_colors = BrandingConfig(
        primary_color="#FF0000",
        secondary_color="#00FF00",
        accent_color="#0000FF",
        text_color="#000000",
        background_color="#FFFFFF",
        primary_font="Arial",
        secondary_font="Arial"
    )
    su.log_info("   Valid hex colors accepted")
except ValueError as e:
    su.log_error(f"   Error: {e}")

try:
    invalid_colors = BrandingConfig(
        primary_color="red",  # Invalid hex format
        secondary_color="#00FF00",
        accent_color="#0000FF",
        text_color="#000000",
        background_color="#FFFFFF",
        primary_font="Arial",
        secondary_font="Arial"
    )
except ValueError as e:
    su.log_error(f"   Invalid color format correctly rejected: {e}")

## 5. Loading Configurations with HydraConfigManager

Let's see how to load configurations using the HydraConfigManager:

In [ ]:
# Save and load with modern HydraConfigManager methods
import tempfile
from pathlib import Path

with HydraConfigManager() as manager:
    # Use a temp directory for this demo
    tmp_dir = Path(tempfile.mkdtemp())
    
    # Save the user we created above
    saved = manager.save_user(user, profiles_dir=tmp_dir / "users")
    su.log_info(f"User saved: {saved}")
    
    # Save the client we created above
    saved = manager.save_client(client, profiles_dir=tmp_dir / "clients")
    su.log_info(f"Client saved: {saved}")
    
    # Load them back
    loaded_user = manager.load_user("dheerajchand", profiles_dir=tmp_dir / "users")
    if loaded_user:
        su.log_info("Loaded User:")
        su.log_info(f"   Person ID: {loaded_user.person_id}")
        su.log_info(f"   Name: {loaded_user.name}")
        su.log_info(f"   Email: {loaded_user.email}")
        su.log_info(f"   Username: {loaded_user.username}")
        su.log_info(f"   Role: {loaded_user.role}")
        su.log_info(f"   Assigned Clients: {loaded_user.get_assigned_clients()}")
        su.log_info(f"   Primary Client: {loaded_user.primary_client}")
    
    loaded_client = manager.load_client("HILL", profiles_dir=tmp_dir / "clients")
    if loaded_client:
        su.log_info("Loaded Client:")
        su.log_info(f"   Person ID: {loaded_client.person_id}")
        su.log_info(f"   Name: {loaded_client.name}")
        su.log_info(f"   Client Code: {loaded_client.client_code}")
        su.log_info(f"   Industry: {loaded_client.industry}")
        su.log_info(f"   Organization: {loaded_client.primary_organization}")
        su.log_info(f"   Brand Primary Color: {loaded_client.branding_config.primary_color}")
        su.log_info(f"   Assigned Users: {loaded_client.get_assigned_users()}")
        su.log_info(f"   Primary User: {loaded_client.primary_user}")
    
    # Clean up temp directory
    import shutil
    shutil.rmtree(tmp_dir)

---

## Part 2: Person / Actor architecture

The underlying data model. Individuals become `Person` records, organizations become `Actor` records, and a `User` or `Client` profile is the projection of one of those into the reporting system.


> **See also:** `foundations/02_profiles_branding.ipynb` covers the practical workflow of creating User and Client profiles. This notebook provides the complete architectural reference for the Person/Actor model system.


In [ ]:
# Import Person/Actor architecture modules
import sys
from pathlib import Path
import siege_utilities as su

# Initialize logging
su.configure_shared_logging(level="INFO")

# Add parent directory to path for notebook imports
notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    sys.path.insert(0, str(notebook_dir.parent))

# Import Person-based models
from siege_utilities.config import (
    Person, User, Client, Collaborator, Organization, Collaboration,
    Credential, OnePasswordCredential, OAuthIntegration, OAuthScope, DatabaseConnection
)

# Note: We'll use the Person/Actor models directly instead of the enhanced_config functions
# which still use the old UserProfile/ClientProfile models

su.log_info("Person/Actor architecture modules imported successfully")

# --- Output configuration ---
OUTPUT_DIR = Path("output") / "notebook_03"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

su.log_info(f"Output directory: {OUTPUT_DIR}  (exists={OUTPUT_DIR.exists()})")

In [ ]:
# 1. Create Organizations
#
# Organization Field Constraints (Pydantic validated):
#   org_id:        1-50 chars, pattern: [a-zA-Z0-9_-]+
#   name:          1-100 chars
#   org_type:      vendor | client | partner | internal
#   primary_email: must match ^[^@]+@[^@]+\.[^@]+$
#   phone:         optional, 10-15 digits (extracted from string)
#   website:       optional, must match ^https?://...
#   address:       optional, max 200 chars
#   status:        active | inactive | archived (default: active)

# Siege Analytics organization
siege_org = Organization(
    org_id='siege_analytics',              # 1-50 chars, [a-zA-Z0-9_-]+
    name='Siege Analytics',                # 1-100 chars
    org_type='internal',                   # vendor | client | partner | internal
    primary_email='dheeraj@siegeanalytics.com',  # valid email format
    phone='+1-512-850-5473',               # optional, 10-15 digits
    website='https://siegeanalytics.com',  # optional, https?:// URL
    address='Austin, TX'                   # optional, max 200 chars
)

# Masai Interactive organization  
masai_org = Organization(
    org_id='masai_interactive',            # 1-50 chars, [a-zA-Z0-9_-]+
    name='Masai Interactive',              # 1-100 chars
    org_type='partner',                    # vendor | client | partner | internal
    primary_email='info@masaiinteractive.com',
    phone=None,
    website='https://masaiinteractive.com',
    address='Washington, DC'
)

# Hillcrest organization
hillcrest_org = Organization(
    org_id='hillcrest_children',           # 1-50 chars, [a-zA-Z0-9_-]+
    name='Hillcrest Children & Family Center',
    org_type='client',                     # vendor | client | partner | internal
    primary_email='info@hillcrestchildren.org',
    phone='+1-202-555-0123',
    website='https://www.hillcrestchildren.org',
    address='Washington, DC'
)

su.log_info("Created organizations:")
su.log_info(f"   - {siege_org.name} ({siege_org.org_type})")
su.log_info(f"   - {masai_org.name} ({masai_org.org_type})")
su.log_info(f"   - {hillcrest_org.name} ({hillcrest_org.org_type})")

In [ ]:
# 2. Create Users with Credentials
#
# Credential Field Constraints (Pydantic validated):
#   name:            1-100 chars, pattern: [a-zA-Z0-9_-]+
#   credential_type: api_key | oauth_token | username_password | ssh_key | certificate | secret
#   service:         1-50 chars, pattern: [a-zA-Z0-9_-]+
#   api_key:         min 10 chars (if provided)
#   password:        min 8 chars (if provided)
#   status:          active | expired | revoked | pending (default: active)
#
# DatabaseConnection Field Constraints (Pydantic validated):
#   name:            1-50 chars, pattern: [a-zA-Z0-9_-]+
#   connection_type: postgresql | mysql | sqlite | duckdb
#   host:            1-255 chars, pattern: [a-zA-Z0-9.-]+
#   port:            1-65535
#   database:        1-50 chars, must start with letter, pattern: [a-zA-Z][a-zA-Z0-9_-]*
#   username:        1-50 chars
#   password:        8-100 chars, REQUIRES: uppercase + lowercase + number
#
# User Field Constraints (extends Person):
#   username:        1-50 chars, pattern: [a-zA-Z0-9_-]+
#   github_login:    optional, max 50 chars, pattern: [a-zA-Z0-9_-]+
#   role:            analyst | manager | developer | admin | collaborator
#
# Collaborator Field Constraints (extends Person):
#   external_organization: 1-100 chars (cannot be empty)
#   collaboration_level:   read | write | admin (default: read)
#   access_expires:        must be in the future (if provided)

from datetime import datetime, timedelta

# Dheeraj (Siege User) with credentials
dheeraj_ga_cred = Credential(
    name='dheeraj_google_analytics',       # 1-100 chars, [a-zA-Z0-9_-]+
    credential_type='api_key',             # api_key | oauth_token | username_password | ...
    service='google_analytics',            # 1-50 chars, [a-zA-Z0-9_-]+
    api_key='GA-123456789-DHEERAJ',        # min 10 chars
    username='dheeraj@siegeanalytics.com'
)

dheeraj_db_conn = DatabaseConnection(
    name='dheeraj_localpostgis',           # 1-50 chars, [a-zA-Z0-9_-]+
    connection_type='postgresql',          # postgresql | mysql | sqlite | duckdb
    host='localhost',                      # 1-255 chars, [a-zA-Z0-9.-]+
    port=5432,                             # 1-65535
    database='postgres',                   # 1-50 chars, starts with letter
    username='dheeraj',                    # 1-50 chars
    password='Dessert123!'                 # 8-100 chars, uppercase + lowercase + number
)

dheeraj = User(
    person_id='dheeraj_siege',             # 1-50 chars, [a-zA-Z0-9_-]+
    name='Dheeraj Chand',                  # 1-100 chars
    email='dheeraj@siegeanalytics.com',    # valid email format
    username='dheerajchand',               # 1-50 chars, [a-zA-Z0-9_-]+
    github_login='dheerajchand',           # optional, [a-zA-Z0-9_-]+
    role='admin',                          # analyst | manager | developer | admin | collaborator
    organizations=['siege_analytics'],
    primary_organization='siege_analytics',
    credentials=[dheeraj_ga_cred],
    database_connections=[dheeraj_db_conn]
)

# Tony (Masai Collaborator) with limited access
tony_oauth = OAuthIntegration(
    name='google_analytics_oauth',         # 1-100 chars, [a-zA-Z0-9_-]+
    provider='google',                     # google | microsoft | github | linkedin | ...
    service='google_analytics',            # 1-50 chars, [a-zA-Z0-9_-]+
    client_id='tony_google_client_123456789',    # 10-200 chars
    client_secret='tony_secret_123456789',       # 10-200 chars
    redirect_uri='https://masaiinteractive.com/oauth',  # must match https?://...
    scopes=[OAuthScope.ANALYTICS]
)

# Use relative future date so this notebook doesn't rot
tony_access_expiry = datetime.now() + timedelta(days=365)

tony = Collaborator(
    person_id='tony_masai',                # 1-50 chars, [a-zA-Z0-9_-]+
    name='Tony Teat',                      # 1-100 chars
    email='tony@masaiinteractive.com',     # valid email format
    external_organization='Masai Interactive',  # 1-100 chars (required)
    collaboration_level='write',           # read | write | admin
    organizations=['masai_interactive'],
    oauth_integrations=[tony_oauth],
    allowed_services=['google_analytics', 'facebook_insights'],
    access_expires=tony_access_expiry      # must be in the future
)

su.log_info("Created users:")
su.log_info(f"   - {dheeraj.name} (User) - {dheeraj.role}")
su.log_info(f"   - {tony.name} (Collaborator) - {tony.collaboration_level}")

In [ ]:
# 3. Create Client with Branding
#
# Client Field Constraints (extends Person):
#   client_code:    2-10 chars, pattern: [A-Z0-9]+ (must be uppercase)
#                   Reserved codes: DEFAULT, ADMIN, SYSTEM, TEST
#   industry:       1-50 chars
#   project_count:  >= 0
#   client_status:  active | inactive | archived
#   branding_config:    optional BrandingConfig (or Dict that matches BrandingConfig schema)
#   report_preferences: optional ReportPreferences (or Dict that matches ReportPreferences schema)

from siege_utilities.config.models.branding_config import BrandingConfig
from siege_utilities.config.models.report_preferences import ReportPreferences

# Hillcrest client with comprehensive branding using typed models
hillcrest_branding = BrandingConfig(
    primary_color='#2E8B57',               # Sea Green
    secondary_color='#4169E1',             # Royal Blue
    accent_color='#FFD700',                # Gold
    text_color='#333333',                  # Dark gray text
    background_color='#FAFAFA',            # Off-white background
    primary_font='Georgia',                # Serif for headings
    secondary_font='Arial',               # Sans-serif for body
    logo_path='hillcrest_logo.png',        # Logo file path
    chart_color_palette='YlGnBu',          # Blue-green palette for charts
)

hillcrest_report_prefs = ReportPreferences(
    default_format='pdf',
    chart_style='professional',
    page_size='Letter',
    include_executive_summary=True,
    include_table_of_contents=True,
)

hillcrest_client = Client(
    person_id='hillcrest_client',          # 1-50 chars, [a-zA-Z0-9_-]+
    name='Hillcrest Children & Family Center',  # 1-100 chars
    email='info@hillcrestchildren.org',    # valid email format
    client_code='HILL',                    # 2-10 chars, [A-Z0-9]+, uppercase only
    industry='Nonprofit - Children & Family Services',  # 1-50 chars
    phone='+1-202-555-0123',               # optional, 10-15 digits
    address='Washington, DC',              # optional, max 200 chars
    website='https://www.hillcrestchildren.org',  # optional, https?://...
    linkedin='https://linkedin.com/company/hillcrest-children-family-center',  # linkedin URL pattern
    project_count=1,                       # >= 0
    client_status='active',                # active | inactive | archived
    organizations=['hillcrest_children'],
    primary_organization='hillcrest_children',
    branding_config=hillcrest_branding,    # Typed BrandingConfig model
    report_preferences=hillcrest_report_prefs,  # Typed ReportPreferences model
)

su.log_info("Created client:")
su.log_info(f"   - {hillcrest_client.name} ({hillcrest_client.industry})")
su.log_info(f"   - Website: {hillcrest_client.website}")
su.log_info(f"   - Location: {hillcrest_client.address}")
su.log_info(f"   - Branding: {type(hillcrest_client.branding_config).__name__}")
su.log_info(f"   - Primary color: {hillcrest_client.branding_config.primary_color}")
su.log_info(f"   - Report format: {hillcrest_client.report_preferences.default_format}")

In [ ]:
# 4. Create Collaboration
#
# Collaboration Field Constraints (Pydantic validated):
#   collab_id:      1-50 chars, pattern: [a-zA-Z0-9_-]+
#   name:           1-100 chars
#   description:    optional, max 500 chars
#   organizations:  List[str], must be unique
#   clients:        List[str], must be unique
#   participants:   List[str], must be unique
#   status:         planning | active | on_hold | completed | cancelled (default: planning)
#   start_date:     datetime (default: now)
#   end_date:       optional, must be in the future

# Hillcrest Analytics Project - Joint project between Siege and Masai
collab_end_date = datetime.now() + timedelta(days=365)

hillcrest_collaboration = Collaboration(
    collab_id='hillcrest_analytics_2025',  # 1-50 chars, [a-zA-Z0-9_-]+
    name='Hillcrest Analytics Project 2025',  # 1-100 chars
    description='Marketing analytics project for Hillcrest Children & Family Center',  # max 500 chars
    organizations=['siege_analytics', 'masai_interactive'],  # unique list
    clients=['hillcrest_children'],        # unique list
    participants=['dheeraj_siege', 'tony_masai'],  # unique list
    status='active',                       # planning | active | on_hold | completed | cancelled
    start_date=datetime(2025, 1, 1),
    end_date=collab_end_date,              # must be in the future
    shared_credentials=['hillcrest_google_analytics', 'hillcrest_facebook_insights'],
    shared_databases=['hillcrest_analytics_db']
)

su.log_info("Created collaboration:")
su.log_info(f"   - {hillcrest_collaboration.name}")
su.log_info(f"   - Organizations: {', '.join(hillcrest_collaboration.organizations)}")
su.log_info(f"   - Participants: {', '.join(hillcrest_collaboration.participants)}")
su.log_info(f"   - Status: {hillcrest_collaboration.status}")

In [ ]:
# 5. Summary of Created Entities

su.log_info("All entities created successfully!")
su.log_info(f"   Organizations: {len([siege_org, masai_org, hillcrest_org])} created")
su.log_info(f"   Users: {len([dheeraj])} created")
su.log_info(f"   Collaborators: {len([tony])} created")
su.log_info(f"   Clients: {len([hillcrest_client])} created")
su.log_info(f"   Collaborations: {len([hillcrest_collaboration])} created")

su.log_info("Entity Summary:")
su.log_info(f"   - {siege_org.name} ({siege_org.org_type})")
su.log_info(f"   - {masai_org.name} ({masai_org.org_type})")
su.log_info(f"   - {hillcrest_org.name} ({hillcrest_org.org_type})")
su.log_info(f"   - {dheeraj.name} (User) - {dheeraj.role}")
su.log_info(f"   - {tony.name} (Collaborator) - {tony.collaboration_level}")
su.log_info(f"   - {hillcrest_client.name} (Client) - {hillcrest_client.industry}")
su.log_info(f"   - {hillcrest_collaboration.name} (Collaboration) - {hillcrest_collaboration.status}")

In [ ]:
# 6. Demonstrate Credential Management
#
# Credential Field Constraints (Pydantic validated):
#   name:            1-100 chars, pattern: [a-zA-Z0-9_-]+
#   credential_type: api_key | oauth_token | username_password | ssh_key | certificate | secret
#   service:         1-50 chars, pattern: [a-zA-Z0-9_-]+
#   api_key:         min 10 chars (if provided)
#
# OnePasswordCredential Field Constraints (Pydantic validated):
#   vault_id:        1-50 chars, pattern: [a-zA-Z0-9_-]+
#   item_id:         1-50 chars, pattern: [a-zA-Z0-9_-]+
#   title:           1-100 chars
#   credential_name: 1-100 chars
#   service:         1-50 chars
#   auto_sync:       bool (default: True)

# Add shared credentials for the Hillcrest project
hillcrest_ga_cred = Credential(
    name='hillcrest_google_analytics',     # 1-100 chars, [a-zA-Z0-9_-]+
    credential_type='api_key',             # api_key | oauth_token | ...
    service='google_analytics',            # 1-50 chars, [a-zA-Z0-9_-]+
    api_key='GA-987654321-HILLCREST',      # min 10 chars
    username='hillcrest@analytics.com'
)

# Add credentials to both users
dheeraj.add_credential(hillcrest_ga_cred)
tony.add_credential(hillcrest_ga_cred)

# Add 1Password credential reference
onepassword_cred = OnePasswordCredential(
    credential_name='hillcrest_facebook_token',  # 1-100 chars
    vault_id='hillcrest_vault',            # 1-50 chars, [a-zA-Z0-9_-]+
    item_id='item_123456789',              # 1-50 chars, [a-zA-Z0-9_-]+
    title='Hillcrest Facebook Token',      # 1-100 chars
    service='Facebook Business',           # 1-50 chars
    auto_sync=True                         # bool
)

dheeraj.add_onepassword_credential(onepassword_cred)

su.log_info("Credential management demonstrated:")
su.log_info(f"   Dheeraj credentials: {len(dheeraj.credentials)}")
su.log_info(f"   Tony credentials: {len(tony.credentials)}")
su.log_info(f"   Dheeraj 1Password refs: {len(dheeraj.onepassword_credentials)}")

# Show credential details
for cred in dheeraj.credentials:
    su.log_info(f"   - {cred.name} ({cred.service})")

In [ ]:
# 6b. Demo: Pull real OAuth2 credential from 1Password
#
# This demonstrates get_google_oauth_from_1password() which retrieves
# the actual GA4 OAuth2 client credentials from 1Password.
# Falls back gracefully when 1Password is unavailable (headless, no op CLI, etc.)

try:
    from siege_utilities.config import get_google_oauth_from_1password
    real_creds = get_google_oauth_from_1password(
        vault='Private', account='Dheeraj_Chand_Family'
    )
    if real_creds:
        su.log_info("Real GA4 OAuth2 credential retrieved from 1Password")
        su.log_info(f"  client_id: {real_creds['client_id'][:20]}...")
        su.log_info(f"  project_id: {real_creds.get('project_id', 'N/A')}")
    else:
        su.log_info("1Password not available — using demo credentials above")
except Exception as e:
    su.log_info(f"1Password not available ({e}) — using demo credentials above")

In [ ]:
# 7. Demonstrate User↔Client Assignment, YAML I/O, and Organization Relationships

# --- User↔Client Bidirectional Assignment ---
su.log_info("Testing User↔Client Assignment:")

# Assign Dheeraj to Hillcrest client
dheeraj.assign_client('HILL')
su.log_info(f"   Dheeraj assigned clients: {dheeraj.get_assigned_clients()}")
su.log_info(f"   Has HILL: {dheeraj.has_client('HILL')}")

# Set primary client
dheeraj.set_primary_client('HILL')
su.log_info(f"   Primary client: {dheeraj.primary_client}")

# Assign user to client (reverse direction)
hillcrest_client.assign_user('dheeraj_siege')
hillcrest_client.set_primary_user('dheeraj_siege')
su.log_info(f"   Hillcrest users: {hillcrest_client.get_assigned_users()}")
su.log_info(f"   Hillcrest primary user: {hillcrest_client.primary_user}")

# --- YAML Serialization ---
su.log_info("\nTesting YAML Serialization:")
yaml_str = dheeraj.to_yaml()
su.log_info(f"   User YAML length: {len(yaml_str)} chars")

# Round-trip
from siege_utilities.config.models.actor_types import User as UserModel
dheeraj_copy = UserModel.from_yaml(yaml_str)
su.log_info(f"   Round-trip: {dheeraj_copy.person_id} == {dheeraj.person_id}")

# Safe export (redact sensitive data) — for sharing, not for re-import
safe_yaml = dheeraj.to_yaml(exclude_sensitive=True)
su.log_info(f"   Safe YAML (no secrets): {len(safe_yaml)} chars")
assert '***REDACTED***' in safe_yaml, "Sensitive data should be redacted"
su.log_info("   Credentials redacted in safe export: PASS")

# --- Bulk Export/Import ---
su.log_info("\nTesting Bulk Export/Import:")
from siege_utilities.config.models.export import export_entities, import_entities

# Full round-trip (no redaction — preserves all data for re-import)
yaml_export = export_entities(
    users=[dheeraj],
    clients=[hillcrest_client],
    organizations=[siege_org, masai_org, hillcrest_org],
    collaborations=[hillcrest_collaboration],
)
su.log_info(f"   Exported {len(yaml_export)} chars of YAML")

result = import_entities(yaml_export)
su.log_info(f"   Imported: {len(result['users'])} users, {len(result['clients'])} clients, "
            f"{len(result['organizations'])} orgs, {len(result['collaborations'])} collabs")
su.log_info(f"   Round-trip user: {result['users'][0].person_id}")

# Safe export for sharing (redacted — not for re-import)
safe_export = export_entities(users=[dheeraj], exclude_sensitive=True)
su.log_info(f"   Safe export: {len(safe_export)} chars (credentials redacted)")

# --- 1Password and Organization Methods ---
su.log_info("\nTesting 1Password Credential Methods:")
has_fb_1p = dheeraj.has_onepassword_credential('hillcrest_facebook_token')
su.log_info(f"   Has Facebook 1Password: {has_fb_1p}")

services = dheeraj.get_onepassword_services()
su.log_info(f"   1Password services: {services}")

coverage = dheeraj.get_credential_coverage()
su.log_info(f"   Total services covered: {coverage['summary']['total_services']}")

recommendations = dheeraj.get_security_recommendations()
su.log_info(f"   Security recommendations: {len(recommendations)}")
for rec in recommendations:
    su.log_info(f"     - {rec}")

su.log_info("\nTesting Organization Relationships:")
su.log_info(f"   Has siege_analytics: {dheeraj.has_organization('siege_analytics')}")

dheeraj.add_collaboration('hillcrest_analytics_2025')
su.log_info(f"   In Hillcrest project: {dheeraj.has_collaboration('hillcrest_analytics_2025')}")

summary = dheeraj.get_summary()
su.log_info(f"   User summary: {summary}")

su.log_info("\nAll Person/Actor functionality demonstrated successfully!")

## Person/Actor Architecture Complete!

### **What We Built:**

#### **1. Organizations**
- **Siege Analytics**: Internal organization
- **Masai Interactive**: Partner organization  
- **Hillcrest Children & Family Center**: Client organization

#### **2. Users & Collaborators**
- **Dheeraj (User)**: Siege admin with full credentials and database access
- **Tony (Collaborator)**: Masai team member with limited access and expiration

#### **3. Client Management with Typed Branding**
- **Hillcrest Client**: Complete with **typed BrandingConfig** and **ReportPreferences** models
- No more loose dicts — branding is validated at creation time (hex colors, font names, etc.)

#### **4. User↔Client Bidirectional Assignment**
- `dheeraj.assign_client('HILL')` / `dheeraj.set_primary_client('HILL')`
- `hillcrest_client.assign_user('dheeraj_siege')` / `hillcrest_client.set_primary_user('dheeraj_siege')`

#### **5. YAML Import/Export**
- Individual entities: `user.to_yaml()` / `User.from_yaml(yaml_str)`
- Bulk export/import: `export_entities(users=[...], clients=[...])` → `import_entities(yaml_str)`
- Sensitive data redaction: `exclude_sensitive=True`

#### **6. Collaboration**
- **Hillcrest Analytics Project**: Joint project between Siege and Masai
- **Shared Resources**: Credentials, databases, and access controls

#### **7. Credential Management**
- **API Keys**: Google Analytics credentials
- **OAuth Integration**: Google OAuth for Tony
- **1Password Integration**: Secure credential references
- **Database Connections**: PostgreSQL access for Dheeraj
- **Coverage Analysis**: Security recommendations based on credential audit

### **Key Benefits:**
- **Multi-Company Support**: Perfect for Siege + Masai collaborations
- **Secure Credential Sharing**: Controlled access to sensitive data
- **Flexible Relationships**: Users ↔ Clients, Users ↔ Organizations
- **Type-Safe Configuration**: Pydantic validation catches errors at creation time
- **YAML Round-Trip**: Export, share, and import entity configurations

---

## Part 3: Branding — colors, fonts, logos, templates

Everything the report / deck / PDF pipelines read when they ask "how should this look for client X?". Comes with predefined templates (Siege Analytics, Masai, Hillcrest) and validates unknown overrides.


In [1]:
import os
import sys
from pathlib import Path
from datetime import datetime
from IPython.display import display

import siege_utilities as su
su.configure_shared_logging(level="INFO")

sys.path.insert(0, str(Path.cwd().parent))

print(f"Python: {sys.version}")
print(f"Platform: {sys.platform}")

# --- Output configuration ---
OUTPUT_DIR = Path("output") / "notebook_10"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

Python: 3.11.11 (main, Jan 23 2025, 20:04:51) [GCC 13.3.0]
Platform: linux
Output directory: output/notebook_10


## 1. Check Credential Backends

Check which credential storage backends are available.

In [2]:
from siege_utilities.config import CredentialManager, credential_status

status = credential_status()

print("Credential Backend Status:")
for backend, info in status.items():
    symbol = "OK" if info['available'] else "MISSING"
    print(f"  [{symbol}] {backend}: {info['status']}")
    print(f"         {info['description']}")

[siege_utilities] 2026-03-04 12:00:26,360 INFO: Available credential backends: ['files', 'env', 'prompt', '1password']


[siege_utilities] 2026-03-04 12:00:26,361 INFO: Credential manager initialized with backends: {'files': True, 'env': True, 'prompt': True, '1password': True, 'keychain': False}


[siege_utilities] 2026-03-04 12:00:26,362 INFO: Credential search paths: [PosixPath('/home/dheerajchand/git/siege-analytics/siege_utilities/notebooks/credentials'), PosixPath('/home/dheerajchand/.siege_utilities/credentials')]


Credential Backend Status:
  [OK] env: Always available
         Environment variables
  [OK] 1password: Authenticated (6 accounts)
         1Password CLI
  [MISSING] keychain: Not available
         Apple Keychain (requires macOS)
  [OK] prompt: Available (fallback)
         Interactive prompts


In [3]:
# Initialize credential manager
cred_manager = CredentialManager()

print(f"Backend priority: {cred_manager.backend_priority}")
print(f"Available backends: {[k for k,v in cred_manager.available_backends.items() if v]}")
print(f"Default vault: {cred_manager.default_vault}")

[siege_utilities] 2026-03-04 12:00:26,394 INFO: Available credential backends: ['files', 'env', 'prompt', '1password']


[siege_utilities] 2026-03-04 12:00:26,394 INFO: Credential manager initialized with backends: {'files': True, 'env': True, 'prompt': True, '1password': True, 'keychain': False}


[siege_utilities] 2026-03-04 12:00:26,394 INFO: Credential search paths: [PosixPath('/home/dheerajchand/git/siege-analytics/siege_utilities/notebooks/credentials'), PosixPath('/home/dheerajchand/.siege_utilities/credentials')]


Backend priority: ['files', 'env', '1password', 'keychain', 'prompt']
Available backends: ['files', 'env', 'prompt', '1password']
Default vault: Private


## 2. Create User Profile

Create or load user profile with default preferences.

In [4]:
# Modern Person/Actor imports
from siege_utilities.config import User, Client, BrandingConfig, ReportPreferences, HydraConfigManager

manager = HydraConfigManager()

profiles_dir = Path.home() / ".siege_utilities" / "profiles"
users_dir = profiles_dir / "users"
clients_dir = profiles_dir / "clients"

print(f"Users dir:   {users_dir} (exists: {users_dir.exists()})")
print(f"Clients dir: {clients_dir} (exists: {clients_dir.exists()})")

if users_dir.exists():
    existing_users = list(users_dir.glob("*.yaml"))
    print(f"Existing user profiles: {[f.stem for f in existing_users]}")

if clients_dir.exists():
    existing_clients = list(clients_dir.glob("*.yaml"))
    print(f"Existing client profiles: {[f.stem for f in existing_clients]}")

[siege_utilities] 2026-03-04 12:00:26,401 INFO: Initialized HydraConfigManager with config directory: /home/dheerajchand/git/siege-analytics/siege_utilities/siege_utilities/configs


Users dir:   /home/dheerajchand/.siege_utilities/profiles/users (exists: True)
Clients dir: /home/dheerajchand/.siege_utilities/profiles/clients (exists: True)
Existing user profiles: ['dheeraj']
Existing client profiles: ['HILL']


In [5]:
# Create user using modern User model
user = User(
    person_id="dheeraj",
    name="Dheeraj Chand",
    email="dheeraj@siegeanalytics.com",
    username="dheeraj",
    github_login="dheerajchand",
    role="admin",
    preferred_download_directory=str(Path.home() / "Downloads" / "siege_utilities"),
    default_output_format="pdf",
    preferred_map_style="carto-positron",
    default_color_scheme="viridis",
    default_dpi=300,
    default_figure_size=(10, 8),
    enable_logging=True,
    log_level="INFO"
)

print(f"User created: {user.person_id} ({user.name})")
print(f"  Username: {user.username}")
print(f"  Default output: {user.default_output_format}")

User created: dheeraj (Dheeraj Chand)
  Username: dheeraj
  Default output: pdf


In [6]:
# Save and verify user profile
success = manager.save_user(user, profiles_dir=users_dir)
print(f"User profile saved: {success}")

loaded_user = manager.load_user("dheeraj", profiles_dir=users_dir)
if loaded_user:
    print(f"Verified: loaded user {loaded_user.person_id} ({loaded_user.name})")
else:
    print("WARNING: Could not load saved user profile")

[siege_utilities] 2026-03-04 12:00:26,415 INFO: Saved modern User: dheeraj


User profile saved: True
[siege_utilities] 2026-03-04 12:00:26,419 INFO: Loaded modern User: dheeraj


Verified: loaded user dheeraj (Dheeraj Chand)


## 3. Create Hillcrest Client Profile

Create a complete client profile with branding configuration.

In [7]:
# Create Hillcrest branding config
hillcrest_branding = BrandingConfig(
    primary_color="#1a4d2e",      # Forest green (placeholder)
    secondary_color="#4a7c59",    # Sage green (placeholder)
    accent_color="#f4a261",       # Warm accent (placeholder)
    text_color="#2d3436",
    background_color="#ffffff",
    primary_font="Helvetica",
    secondary_font="Georgia",
    logo_path=None,
    header_height=50, footer_height=25,
    margin_top=25, margin_bottom=25, margin_left=20, margin_right=20,
    title_font_size=28, subtitle_font_size=20, body_font_size=12, caption_font_size=10,
    chart_color_palette="viridis",
    chart_background_color="#ffffff",
    chart_grid_color="#e0e0e0"
)

print(f"Branding: {hillcrest_branding.primary_color} / {hillcrest_branding.secondary_color}")
print(f"Fonts: {hillcrest_branding.primary_font} / {hillcrest_branding.secondary_font}")
print(f"Color scheme: {hillcrest_branding.get_color_scheme()}")

Branding: #1a4d2e / #4a7c59
Fonts: Helvetica / Georgia
Color scheme: {'primary': '#1a4d2e', 'secondary': '#4a7c59', 'accent': '#f4a261', 'text': '#2d3436', 'background': '#ffffff', 'chart_background': '#ffffff', 'chart_grid': '#e0e0e0'}


In [8]:
# Create Hillcrest report preferences
hillcrest_reports = ReportPreferences(
    default_format="pdf",
    include_executive_summary=True,
    chart_style="professional",
    page_size="Letter",
    orientation="landscape",
    include_table_of_contents=True,
    include_page_numbers=True,
    chart_quality="high",
    chart_dpi=300,
    include_chart_titles=True,
    include_chart_legends=True,
    sections=["executive_summary", "methodology", "findings", "recommendations"]
)

print(f"Report: {hillcrest_reports.default_format}, {hillcrest_reports.page_size} {hillcrest_reports.orientation}")
print(f"Sections: {hillcrest_reports.sections}")

Report: ReportFormat.PDF, Letter PageOrientation.LANDSCAPE
Sections: ['executive_summary', 'methodology', 'findings', 'recommendations']


In [9]:
# Hillcrest contact info
hillcrest_email = "contact@hillcrest.example.com"
hillcrest_phone = "(555) 123-4567"
hillcrest_website = "https://www.hillcrest.example.com"

print(f"Email:   {hillcrest_email}")
print(f"Phone:   {hillcrest_phone}")
print(f"Website: {hillcrest_website}")

Email:   contact@hillcrest.example.com
Phone:   (555) 123-4567
Website: https://www.hillcrest.example.com


In [10]:
# Create the complete Hillcrest client
hillcrest = Client(
    person_id="hillcrest",
    name="Hillcrest",
    email=hillcrest_email,
    phone=hillcrest_phone,
    website=hillcrest_website,
    client_code="HILL",
    industry="Political Consulting",
    project_count=0,
    client_status="active",
    branding_config=hillcrest_branding,
    report_preferences=hillcrest_reports,
    notes="Primary client for Siege Analytics"
)

# Bidirectional user-client assignment
hillcrest.assign_user(user.person_id)
hillcrest.set_primary_user(user.person_id)
user.assign_client(hillcrest.client_code)
user.set_primary_client(hillcrest.client_code)

print(f"Client created: {hillcrest.name} ({hillcrest.client_code})")
print(f"  Industry: {hillcrest.industry}")
print(f"  Status: {hillcrest.client_status}")
print(f"  Brand color: {hillcrest.branding_config.primary_color}")
print(f"  Assigned users: {hillcrest.get_assigned_users()}")
print(f"  Primary user: {hillcrest.primary_user}")

Client created: Hillcrest (HILL)
  Industry: Political Consulting
  Status: active
  Brand color: #1a4d2e
  Assigned users: ['dheeraj']
  Primary user: dheeraj


In [11]:
# Save and verify Hillcrest profile
success = manager.save_client(hillcrest, profiles_dir=clients_dir)
print(f"Hillcrest profile saved: {success}")

manager.save_user(user, profiles_dir=users_dir)

loaded_hillcrest = manager.load_client("HILL", profiles_dir=clients_dir)
if loaded_hillcrest:
    print(f"Verified: loaded client {loaded_hillcrest.name}")
    print(f"  Code: {loaded_hillcrest.client_code}")
    print(f"  Primary color: {loaded_hillcrest.branding_config.primary_color}")
    print(f"  Assigned users: {loaded_hillcrest.get_assigned_users()}")
else:
    print("WARNING: Could not load saved client profile")

[siege_utilities] 2026-03-04 12:00:26,451 INFO: Saved modern Client: HILL


Hillcrest profile saved: True
[siege_utilities] 2026-03-04 12:00:26,454 INFO: Saved modern User: dheeraj


[siege_utilities] 2026-03-04 12:00:26,459 INFO: Loaded modern Client: HILL


Verified: loaded client Hillcrest
  Code: HILL
  Primary color: #1a4d2e
  Assigned users: ['dheeraj']


## 4. Test 1Password Integration

Retrieve credentials from 1Password (if available).

In [12]:
from siege_utilities.config import get_google_service_account_from_1password

# Check if 1Password is available
if cred_manager.available_backends.get('1password'):
    print("=== 1Password Available ===")
    
    # List stored credentials with siege-utilities tag
    stored_creds = cred_manager.list_stored_credentials()
    print(f"Found {len(stored_creds)} stored credentials (tagged 'siege-utilities'):")
    for cred in stored_creds:
        print(f"  - {cred.get('service', 'unknown')} ({cred.get('backend', 'unknown')})")
    
    if not stored_creds:
        print("  (No items tagged 'siege-utilities' — tag items in 1Password to include them)")
else:
    print("1Password not available - skipping 1Password tests")

=== 1Password Available ===
[siege_utilities] 2026-03-04 12:00:26,489 INFO: 0 items tagged 'siege-utilities' in 1Password (0 total items in vault). Tag items with 'siege-utilities' to include them.


[siege_utilities] 2026-03-04 12:00:26,490 INFO: Found 0 stored credentials


Found 0 stored credentials (tagged 'siege-utilities'):
  (No items tagged 'siege-utilities' — tag items in 1Password to include them)


In [13]:
# Try to get Google Analytics service account from 1Password
# The item title should match what's stored in 1Password

ga_service_account = None

if cred_manager.available_backends.get('1password'):
    # Try common item titles
    item_titles = [
        "Google Analytics Service Account - Multi-Client Reporter",
        "Google Analytics Service Account",
        "GA4 Service Account",
        "Hillcrest GA Service Account"
    ]
    
    for title in item_titles:
        try:
            print(f"Trying: {title}...")
            ga_service_account = get_google_service_account_from_1password(title)
            if ga_service_account:
                print("Found service account!")
                print(f"   Project: {ga_service_account.get('project_id')}")
                print(f"   Email: {ga_service_account.get('client_email')}")
                break
        except Exception as e:
            print(f"   Not found or error: {str(e)[:50]}")
            continue
    
    if not ga_service_account:
        print("WARNING: No GA service account found in 1Password")
        print("To store one, use:")
        print("  from siege_utilities.config import store_ga_service_account_from_file")
        print("  store_ga_service_account_from_file('/path/to/service-account.json')")
else:
    print("Skipping 1Password credential retrieval (not available)")

Trying: Google Analytics Service Account - Multi-Client Reporter...
[siege_utilities] 2026-03-04 12:00:26,510 ERROR: Failed to get Google service account from 1Password: Command '['op', 'item', 'get', 'Google Analytics Service Account - Multi-Client Reporter', '--field=project_id', '--reveal']' returned non-zero exit status 1.


Trying: Google Analytics Service Account...
[siege_utilities] 2026-03-04 12:00:26,521 ERROR: Failed to get Google service account from 1Password: Command '['op', 'item', 'get', 'Google Analytics Service Account', '--field=project_id', '--reveal']' returned non-zero exit status 1.


Trying: GA4 Service Account...
[siege_utilities] 2026-03-04 12:00:26,533 ERROR: Failed to get Google service account from 1Password: Command '['op', 'item', 'get', 'GA4 Service Account', '--field=project_id', '--reveal']' returned non-zero exit status 1.


Trying: Hillcrest GA Service Account...
[siege_utilities] 2026-03-04 12:00:26,544 ERROR: Failed to get Google service account from 1Password: Command '['op', 'item', 'get', 'Hillcrest GA Service Account', '--field=project_id', '--reveal']' returned non-zero exit status 1.


To store one, use:
  from siege_utilities.config import store_ga_service_account_from_file
  store_ga_service_account_from_file('/path/to/service-account.json')


## 5. Generate Branded PDF Report

Create a sample report using Hillcrest branding.

In [14]:
import pandas as pd
import numpy as np

# Create sample data for the report
np.random.seed(42)

# Sample polling data
poll_data = pd.DataFrame({
    'Candidate': ['Smith', 'Jones', 'Williams', 'Undecided'],
    'Support (%)': [42, 38, 12, 8],
    'Previous (%)': [40, 39, 13, 8],
    'Change': ['+2', '-1', '-1', '0']
})

# Sample demographic breakdown
demo_data = pd.DataFrame({
    'Age Group': ['18-29', '30-44', '45-59', '60+'],
    'Smith': [35, 40, 45, 48],
    'Jones': [45, 42, 35, 32],
    'Williams': [15, 12, 12, 10]
})

print("=== Sample Polling Data ===")
display(poll_data)
print("\n=== Demographic Breakdown ===")
display(demo_data)

=== Sample Polling Data ===


,Candidate,Support (%),Previous (%),Change
0,Smith,42,40,+2
1,Jones,38,39,-1
2,Williams,12,13,-1
3,Undecided,8,8,0



=== Demographic Breakdown ===


,Age Group,Smith,Jones,Williams
0,18-29,35,45,15
1,30-44,40,42,12
2,45-59,45,35,12
3,60+,48,32,10


In [15]:
from siege_utilities.reporting.report_generator import ReportGenerator

# Initialize ReportGenerator with Hillcrest branding — output goes to OUTPUT_DIR
rg = ReportGenerator(
    client_name="Hillcrest",
    output_dir=OUTPUT_DIR
)

print("=== ReportGenerator Initialized ===")
print(f"Client: {rg.client_name}")
print(f"Output directory: {rg.output_dir}")

=== ReportGenerator Initialized ===
Client: Hillcrest
Output directory: output/notebook_10


In [16]:
# Build report content
report_content = {
    'sections': [],
    'metadata': {
        'title': 'Sample Polling Report',
        'subtitle': 'Q1 2026 Survey Results',
        'author': 'Siege Analytics',
        'date': datetime.now().strftime('%B %d, %Y'),
        'client': 'Hillcrest'
    }
}

# Add executive summary
executive_summary = """
This polling report presents findings from a survey of 1,200 likely voters 
conducted January 10-15, 2026. Key findings include:

- Smith leads with 42% support, up 2 points from previous poll
- Jones at 38%, down 1 point
- Williams at 12%, essentially unchanged
- 8% remain undecided with 3 weeks until election

Margin of error: +/-2.8 percentage points (95% confidence)
"""

report_content = rg.add_text_section(
    report_content, 
    'Executive Summary', 
    executive_summary
)

print("Added executive summary")

Added executive summary


In [17]:
# Add polling data table
report_content = rg.add_table_section(
    report_content,
    'Current Poll Results',
    poll_data
)

print("Added polling data table")

Added polling data table


In [18]:
# Add demographic breakdown table
report_content = rg.add_table_section(
    report_content,
    'Support by Age Group',
    demo_data
)

print("Added demographic breakdown table")

Added demographic breakdown table


In [19]:
# Add methodology section
methodology = """
Survey Methodology:

- Sample Size: 1,200 likely voters
- Mode: Mixed (60% cell, 40% landline)
- Field Dates: January 10-15, 2026
- Weighting: Age, gender, education, party registration
- Margin of Error: +/-2.8 percentage points

The survey was conducted by Siege Analytics using live interviewers. 
Respondents were selected using stratified random sampling from 
voter file records.
"""

report_content = rg.add_text_section(
    report_content,
    'Methodology',
    methodology
)

print("Added methodology section")

Added methodology section


In [ ]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak, HRFlowable
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from siege_utilities.reporting.chart_generator import ChartGenerator
from siege_utilities.reporting.image_utils import save_rl_image
import matplotlib
matplotlib.use('Agg')

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_path = OUTPUT_DIR / f"hillcrest_poll_report_{timestamp}.pdf"

# --- Hillcrest branding colors ---
hc_primary = colors.HexColor(hillcrest_branding.primary_color)      # #1a4d2e forest green
hc_secondary = colors.HexColor(hillcrest_branding.secondary_color)   # #4a7c59 sage green
hc_accent = colors.HexColor(hillcrest_branding.accent_color)         # #f4a261 warm accent
hc_text = colors.HexColor(hillcrest_branding.text_color)             # #2d3436
hc_white = colors.white
hc_light_bg = colors.HexColor('#f0f7f2')

# Chart generator with Hillcrest palette
hc_branding = {
    'name': 'Hillcrest',
    'colors': {
        'primary': hillcrest_branding.primary_color,
        'secondary': hillcrest_branding.secondary_color,
        'accent': hillcrest_branding.accent_color,
    }
}
chart_gen_hc = ChartGenerator(
    branding_config=hc_branding, output_dir=OUTPUT_DIR,
    max_chart_width=5.5, max_chart_height=3.5,
)
chart_gen_hc.default_dpi = 200

# Chart dimensions for ReportLab
CHART_W = 5.5 * inch
CHART_H = 3.5 * inch

# --- Branded styles ---
styles = getSampleStyleSheet()
styles.add(ParagraphStyle('HC_Title', parent=styles['Title'],
    fontSize=32, textColor=hc_primary, spaceAfter=4))
styles.add(ParagraphStyle('HC_Subtitle', parent=styles['Normal'],
    fontSize=16, textColor=hc_secondary, alignment=TA_CENTER, spaceBefore=8, spaceAfter=20))
styles.add(ParagraphStyle('HC_Heading', parent=styles['Heading1'],
    fontSize=18, textColor=hc_primary, spaceBefore=12, spaceAfter=8))
styles.add(ParagraphStyle('HC_SubHeading', parent=styles['Heading2'],
    fontSize=14, textColor=hc_secondary, spaceBefore=8, spaceAfter=6))
styles.add(ParagraphStyle('HC_Body', parent=styles['Normal'],
    fontSize=11, textColor=hc_text, leading=15, spaceAfter=4))
styles.add(ParagraphStyle('HC_Bullet', parent=styles['Normal'],
    fontSize=11, textColor=hc_text, leading=15, leftIndent=20, spaceAfter=2))
styles.add(ParagraphStyle('HC_Meta', parent=styles['Normal'],
    fontSize=10, textColor=colors.HexColor('#666666'), alignment=TA_CENTER))
styles.add(ParagraphStyle('HC_Caption', parent=styles['Normal'],
    fontSize=9, textColor=colors.HexColor('#888888'), italic=True,
    alignment=TA_CENTER, spaceBefore=4, spaceAfter=8))
styles.add(ParagraphStyle('HC_Callout', parent=styles['Normal'],
    fontSize=12, textColor=hc_primary, bold=True, alignment=TA_CENTER,
    spaceBefore=8, spaceAfter=8))

# =================================================================
# Generate charts
# =================================================================

# 1. Horse race bar chart — current support
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.5), dpi=200)
candidates = poll_data['Candidate'].tolist()
support = poll_data['Support (%)'].tolist()
bar_colors = [hillcrest_branding.primary_color, hillcrest_branding.secondary_color,
              hillcrest_branding.accent_color, '#cccccc']
bars = ax.barh(candidates[::-1], support[::-1], color=bar_colors[::-1], edgecolor='white', height=0.6)
for bar, val in zip(bars, support[::-1]):
    ax.text(bar.get_width() + 0.8, bar.get_y() + bar.get_height()/2,
            f'{val}%', va='center', fontsize=11, fontweight='bold')
ax.set_xlim(0, 55)
ax.set_xlabel('Support (%)')
ax.set_title('Current Candidate Support', fontsize=14, fontweight='bold',
             color=hillcrest_branding.primary_color)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
horse_race_img = chart_gen_hc._matplotlib_to_reportlab_image(fig, width=CHART_W, height=CHART_H)
save_rl_image(horse_race_img, OUTPUT_DIR / 'poll_horse_race.png')
plt.close(fig)

# 2. Change from previous poll — grouped bar
fig, ax = plt.subplots(figsize=(6, 3.5), dpi=200)
x = np.arange(len(candidates))
width = 0.35
bars1 = ax.bar(x - width/2, poll_data['Previous (%)'].tolist(), width,
               label='Previous', color=hillcrest_branding.secondary_color, alpha=0.7)
bars2 = ax.bar(x + width/2, support, width,
               label='Current', color=hillcrest_branding.primary_color)
ax.set_xticks(x)
ax.set_xticklabels(candidates)
ax.set_ylabel('Support (%)')
ax.set_title('Movement Since Last Poll', fontsize=14, fontweight='bold',
             color=hillcrest_branding.primary_color)
ax.legend(loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.0f}%', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
movement_img = chart_gen_hc._matplotlib_to_reportlab_image(fig, width=CHART_W, height=CHART_H)
save_rl_image(movement_img, OUTPUT_DIR / 'poll_movement.png')
plt.close(fig)

# 3. Demographic breakdown — grouped bar by age
fig, ax = plt.subplots(figsize=(6, 3.5), dpi=200)
age_groups = demo_data['Age Group'].tolist()
x = np.arange(len(age_groups))
width = 0.25
ax.bar(x - width, demo_data['Smith'].tolist(), width, label='Smith',
       color=hillcrest_branding.primary_color)
ax.bar(x, demo_data['Jones'].tolist(), width, label='Jones',
       color=hillcrest_branding.secondary_color)
ax.bar(x + width, demo_data['Williams'].tolist(), width, label='Williams',
       color=hillcrest_branding.accent_color)
ax.set_xticks(x)
ax.set_xticklabels(age_groups)
ax.set_ylabel('Support (%)')
ax.set_title('Support by Age Group', fontsize=14, fontweight='bold',
             color=hillcrest_branding.primary_color)
ax.legend(loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
demo_chart_img = chart_gen_hc._matplotlib_to_reportlab_image(fig, width=CHART_W, height=CHART_H)
save_rl_image(demo_chart_img, OUTPUT_DIR / 'poll_demographics.png')
plt.close(fig)

# 4. Pie chart — current share (excluding undecided)
decided = poll_data[poll_data['Candidate'] != 'Undecided'].copy()
decided_pct = decided['Support (%)'].tolist()
decided_names = decided['Candidate'].tolist()
fig, ax = plt.subplots(figsize=(5, 3.5), dpi=200)
wedges, texts, autotexts = ax.pie(
    decided_pct, labels=decided_names, autopct='%1.0f%%',
    colors=[hillcrest_branding.primary_color, hillcrest_branding.secondary_color,
            hillcrest_branding.accent_color],
    startangle=90, textprops={'fontsize': 10},
)
for at in autotexts:
    at.set_color('white')
    at.set_fontweight('bold')
ax.set_title('Decided Voter Share', fontsize=14, fontweight='bold',
             color=hillcrest_branding.primary_color)
plt.tight_layout()
pie_img = chart_gen_hc._matplotlib_to_reportlab_image(fig, width=CHART_W, height=CHART_H)
save_rl_image(pie_img, OUTPUT_DIR / 'poll_decided_share.png')
plt.close(fig)

print(f"Generated 4 polling charts at 200 DPI")

# =================================================================
# Helper: styled table
# =================================================================
def make_table(df, col_widths=None):
    data = [df.columns.tolist()] + df.values.tolist()
    tbl = Table(data, colWidths=col_widths)
    tbl.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), hc_primary),
        ('TEXTCOLOR', (0, 0), (-1, 0), hc_white),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 11),
        ('FONTNAME', (0, 1), (-1, -1), 'Helvetica'),
        ('FONTSIZE', (0, 1), (-1, -1), 10),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('BOTTOMPADDING', (0, 0), (-1, 0), 10),
        ('TOPPADDING', (0, 0), (-1, 0), 8),
        ('BOTTOMPADDING', (0, 1), (-1, -1), 6),
        ('TOPPADDING', (0, 1), (-1, -1), 6),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.HexColor('#cccccc')),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [hc_white, hc_light_bg]),
    ]))
    return tbl

# =================================================================
# Build the PDF
# =================================================================
story = []

# --- Title page ---
story.append(Spacer(1, 2.0 * inch))
story.append(HRFlowable(width='70%', thickness=3, color=hc_primary, spaceAfter=16))
story.append(Paragraph('Hillcrest District Poll', styles['HC_Title']))
story.append(Paragraph('Q1 2026 Likely Voter Survey', styles['HC_Subtitle']))
story.append(HRFlowable(width='70%', thickness=1, color=hc_secondary, spaceBefore=4, spaceAfter=30))
story.append(Paragraph('n = 1,200 Likely Voters | MOE +/-2.8%', styles['HC_Meta']))
story.append(Spacer(1, 8))
story.append(Paragraph('Prepared for: Hillcrest', styles['HC_Meta']))
story.append(Spacer(1, 4))
story.append(Paragraph('Prepared by: Siege Analytics', styles['HC_Meta']))
story.append(Spacer(1, 4))
story.append(Paragraph(f'{datetime.now().strftime("%B %d, %Y")}', styles['HC_Meta']))

# --- Page 2: Top-line findings (chart + table + argument) ---
story.append(PageBreak())
story.append(HRFlowable(width='100%', thickness=2, color=hc_primary, spaceAfter=8))
story.append(Paragraph('The Horse Race', styles['HC_Heading']))
story.append(Spacer(1, 4))

story.append(Paragraph(
    'Smith holds a <b>4-point lead</b> over Jones among likely voters, '
    'outside the margin of error. This represents a net 3-point swing '
    'toward Smith since the previous survey.',
    styles['HC_Body']))
story.append(Spacer(1, 8))

# Horse race chart
story.extend(chart_gen_hc.create_chart_with_caption(
    horse_race_img,
    'Figure 1: Current candidate support among likely voters (n=1,200)'))
story.append(Spacer(1, 12))

# Table
story.append(Paragraph('Detailed Results', styles['HC_SubHeading']))
story.append(Spacer(1, 4))
story.append(make_table(poll_data, [1.5*inch, 1.2*inch, 1.2*inch, 1.0*inch]))

# --- Page 3: Movement + argument ---
story.append(PageBreak())
story.append(HRFlowable(width='100%', thickness=2, color=hc_primary, spaceAfter=8))
story.append(Paragraph('Movement', styles['HC_Heading']))
story.append(Spacer(1, 4))

story.append(Paragraph(
    'Smith gained 2 points while Jones lost 1 point since the previous poll. '
    'Williams is flat at 12%. The undecided pool (8%) is small enough that '
    'even a complete break toward Jones would not close the gap.',
    styles['HC_Body']))
story.append(Spacer(1, 8))

story.extend(chart_gen_hc.create_chart_with_caption(
    movement_img,
    'Figure 2: Previous vs current support — net 3-point swing toward Smith'))
story.append(Spacer(1, 12))

story.append(Paragraph(
    'Among decided voters, Smith commands <b>46%</b> of the share '
    'vs Jones at <b>41%</b> and Williams at <b>13%</b>.',
    styles['HC_Body']))
story.append(Spacer(1, 8))
story.extend(chart_gen_hc.create_chart_with_caption(
    pie_img,
    'Figure 3: Share among decided voters (undecided excluded)'))

# --- Page 4: Demographics + argument ---
story.append(PageBreak())
story.append(HRFlowable(width='100%', thickness=2, color=hc_primary, spaceAfter=8))
story.append(Paragraph('The Demographic Picture', styles['HC_Heading']))
story.append(Spacer(1, 4))

story.append(Paragraph(
    'Smith dominates with voters 45 and older, leading Jones by <b>10+ points</b> '
    'in both the 45-59 and 60+ cohorts. Jones leads among younger voters '
    '(18-29) by 10 points, but this group has historically lower turnout. '
    'The 30-44 cohort is the most competitive, with just 2 points separating '
    'the top two candidates.',
    styles['HC_Body']))
story.append(Spacer(1, 8))

story.extend(chart_gen_hc.create_chart_with_caption(
    demo_chart_img,
    'Figure 4: Support by age group — Smith strongest with 45+, Jones leads 18-29'))
story.append(Spacer(1, 12))

story.append(Paragraph('Breakdown by Age Cohort', styles['HC_SubHeading']))
story.append(Spacer(1, 4))
story.append(make_table(demo_data, [1.2*inch, 1.0*inch, 1.0*inch, 1.0*inch]))

# --- Page 5: Methodology ---
story.append(PageBreak())
story.append(HRFlowable(width='100%', thickness=2, color=hc_primary, spaceAfter=8))
story.append(Paragraph('Methodology', styles['HC_Heading']))
story.append(Spacer(1, 8))

for b in [
    '<b>Sample Size:</b> 1,200 likely voters',
    '<b>Mode:</b> Mixed (60% cell phone, 40% landline)',
    '<b>Field Dates:</b> January 10-15, 2026',
    '<b>Weighting:</b> Age, gender, education, party registration',
    '<b>Margin of Error:</b> +/-2.8 percentage points (95% CI)',
    '<b>Sampling Frame:</b> Voter file, stratified random',
    '<b>Interviewing:</b> Live interviewers',
]:
    story.append(Paragraph(f'&bull; {b}', styles['HC_Bullet']))

story.append(Spacer(1, 12))
story.append(Paragraph(
    'The survey was conducted by Siege Analytics using live interviewers. '
    'Respondents were selected using stratified random sampling from '
    'voter file records. Results are weighted to match the demographic '
    'composition of the likely electorate based on past turnout models.',
    styles['HC_Body']))
story.append(Spacer(1, 12))
story.append(HRFlowable(width='40%', thickness=0.5, color=hc_secondary))
story.append(Spacer(1, 8))
story.append(Paragraph(
    'Siege Analytics | contact@siegeanalytics.com | siegeanalytics.com',
    styles['HC_Meta']))

# --- Build ---
doc = SimpleDocTemplate(
    str(output_path), pagesize=letter,
    topMargin=1*inch, bottomMargin=1*inch,
    leftMargin=1*inch, rightMargin=1*inch,
)
doc.build(story)

size_kb = output_path.stat().st_size / 1024
print(f"Polling report generated: {output_path}")
print(f"Size: {size_kb:.1f} KB, 5 pages")
print(f"Contents: 4 charts, 2 tables, narrative argument per section")

## 6. Verify Profile Persistence

Verify that profiles are correctly saved and can be reloaded.

In [21]:
# Reload and verify all profiles using modern API
print("=== Verification: Reload All Profiles ===")

# User profile
loaded_user = manager.load_user("dheeraj", profiles_dir=users_dir)
if loaded_user:
    print(f"[OK] User: {loaded_user.person_id} ({loaded_user.name})")
    print(f"   Username: {loaded_user.username}")
    print(f"   Default format: {loaded_user.default_output_format}")
    print(f"   Assigned clients: {loaded_user.get_assigned_clients()}")
else:
    print("ERROR: User profile not found")

# Client profile
loaded_client = manager.load_client("HILL", profiles_dir=clients_dir)
if loaded_client:
    print(f"[OK] Client: {loaded_client.name}")
    print(f"   Code: {loaded_client.client_code}")
    print(f"   Industry: {loaded_client.industry}")
    print(f"   Primary color: {loaded_client.branding_config.primary_color}")
    print(f"   Primary font: {loaded_client.branding_config.primary_font}")
    print(f"   Assigned users: {loaded_client.get_assigned_users()}")
else:
    print("ERROR: Client profile not found")

=== Verification: Reload All Profiles ===
[siege_utilities] 2026-03-04 12:00:28,480 INFO: Loaded modern User: dheeraj


[OK] User: dheeraj (Dheeraj Chand)
   Username: dheeraj
   Default format: pdf
   Assigned clients: ['HILL']
[siege_utilities] 2026-03-04 12:00:28,486 INFO: Loaded modern Client: HILL


[OK] Client: Hillcrest
   Code: HILL
   Industry: Political Consulting
   Primary color: #1a4d2e
   Primary font: Helvetica
   Assigned users: ['dheeraj']


In [22]:
# Check saved profile files
print("=== Profile Files ===")

profiles_base = Path.home() / ".siege_utilities" / "profiles"

for profile_type in ["users", "clients"]:
    profile_dir = profiles_base / profile_type
    if profile_dir.exists():
        files = list(profile_dir.glob("*.yaml"))
        print(f"{profile_type.title()}:")
        for f in files:
            size = f.stat().st_size
            print(f"  - {f.name} ({size} bytes)")
    else:
        print(f"WARNING: {profile_type.title()}: Directory not found")

=== Profile Files ===
Users:
  - dheeraj.yaml (961 bytes)
Clients:
  - HILL.yaml (1756 bytes)


## 7. Design Kit Export

Export a complete design kit from the Hillcrest report data using `export_design_kit()`.
This produces a folder structure with separated assets for InDesign/Illustrator handoff:

```
output_dir/
    charts/          SVG vector chart files
    charts-png/      High-resolution PNG chart files (300 DPI)
    tables/          CSV exports of all data tables
    text/            report_text.md with narrative sections
    metadata.yaml    Client name, dates, KPIs, summary stats
```

This is useful when handing off analytics results to a graphic designer who
needs editable vector charts and raw data tables alongside the narrative text.

In [ ]:
import tempfile
from datetime import datetime, timedelta

try:
    from siege_utilities.reporting.examples.google_analytics_report_example import (
        export_design_kit,
        generate_sample_ga_data,
    )
    HAS_DESIGN_KIT = True
except ImportError as e:
    HAS_DESIGN_KIT = False
    print(f"Design kit import failed ({e}) — skipping.")

if HAS_DESIGN_KIT:
    # Generate sample GA data to demonstrate the export
    end_date = datetime.now()
    start_date = end_date - timedelta(days=30)
    ga_data = generate_sample_ga_data(start_date, end_date)
    print("=== Sample GA Data ===")
    print(f"  Total users   : {ga_data['totals']['users']:,}")
    print(f"  Total sessions: {ga_data['totals']['sessions']:,}")
    print(f"  Date range    : {ga_data['date_range']}")
    print(f"  Daily records : {len(ga_data['daily_data'])}")
    print(f"  Sources       : {len(ga_data['traffic_sources'])}")
    print()

    # Export the design kit
    with tempfile.TemporaryDirectory() as tmpdir:
        export_path = export_design_kit(
            ga_data=ga_data,
            output_dir=tmpdir,
        )
        print(f"Design kit exported to: {export_path}")
        if export_path:
            import os
            for f in os.listdir(tmpdir):
                fpath = os.path.join(tmpdir, f)
                size = os.path.getsize(fpath)
                print(f"  {f}: {size:,} bytes")

In [ ]:
if HAS_DESIGN_KIT:
    # Export the design kit to the notebook output directory
    kit_dir = OUTPUT_DIR / "design_kit"

    success = export_design_kit(
        ga_data,
        output_dir=str(kit_dir),
        client_name="Hillcrest",
        prepared_by="Siege Analytics",
    )
    print(f"Design kit export success: {success}")

    if success:
        print(f"\n=== Design Kit Contents ===")
        for item in sorted(kit_dir.rglob("*")):
            rel = item.relative_to(kit_dir)
            if item.is_file():
                print(f"  {rel}  ({item.stat().st_size:,} bytes)")
            else:
                print(f"  {rel}/")
else:
    print("Skipping design kit export (import not available).")

In [ ]:
# Summary of all tests
print("=" * 50)
print("PROFILE & BRANDING SYSTEM TEST RESULTS")
print("=" * 50)

results = {
    "Credential Backends": "1password" in [k for k,v in cred_manager.available_backends.items() if v],
    "User Profile Created": manager.load_user("dheeraj", profiles_dir=users_dir) is not None,
    "Client Profile Created": manager.load_client("HILL", profiles_dir=clients_dir) is not None,
    "1Password Integration": cred_manager.available_backends.get('1password', False),
    "PDF Generation": output_path.exists() if 'output_path' in dir() else False,
    "Design Kit Export": HAS_DESIGN_KIT and kit_dir.exists() if 'kit_dir' in dir() else False,
}

for test, passed in results.items():
    symbol = "PASS" if passed else "FAIL"
    print(f"  {symbol}  {test}")

passed = sum(results.values())
total = len(results)
print(f"\n{passed}/{total} tests passed.")

if passed == total:
    print("\nAll Profile/Branding tests passed!")
else:
    print("\nSome tests need attention.")

# Clean up HydraConfigManager
manager.cleanup()

## Next Steps

If all tests pass:

1. **Update Hillcrest branding colors** - Edit the BrandingConfig with actual Hillcrest colors
2. **Add logo** - Set `logo_path` to actual Hillcrest logo file
3. **Store GA credentials** - If you have GA service account, store it:
   ```python
   from siege_utilities.config import store_ga_service_account_from_file
   store_ga_service_account_from_file('/path/to/service-account.json', 
                                       item_title='Hillcrest GA Service Account')
   ```
4. **Test with real data** - Use actual polling data from Hillcrest projects

## Related
- Source: `siege_utilities/admin/profile_manager.py`, `siege_utilities/reporting/client_branding.py`
- Tests: `tests/test_admin_profile_manager.py`, `tests/test_client_branding*.py`
- Consolidates: previous `03_Person_Actor_Architecture` and `10_Profile_Branding_Testing`
